# 03 · Full benchmark: Laya vs XGBoost vs Random Forest vs BERT

Every model sits behind the same `DecisionModel` interface, sees the **same test set**, and is scored by the
same `run_benchmark` function, so the comparison is apples to apples.

| model | what it is | params |
|---|---|---|
| Random Forest | 300 trees on TF-IDF + one-hot + scaled metadata | n/a (trees) |
| XGBoost | 300 boosted trees, same features | n/a (trees) |
| BERT | `bert-base-uncased`, fine-tuned, metadata written into the text | 110M |
| Laya base | `convaiinnovations/laya`, zero-shot | 421M |
| Laya typed-decisions | authors' fine-tune, zero-shot on our data | 421M |
| Laya fine-tuned | base Laya fine-tuned on our train split (from notebook 02) | 421M |

Metrics: accuracy + macro-F1 per field, **exact match** (all three fields right), model size, latency **p50 / p95**
over 100 single-ticket calls.

**Runtime:** GPU (T4). Classical models run on CPU either way; keep that in mind when reading latency.

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

Install everything (Laya, transformers for BERT, matplotlib for charts).

In [ ]:
%pip install -q laya transformers scikit-learn xgboost pandas matplotlib tabulate
os.environ["USE_TF"] = "0"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")

## 1. Data

Same splits as notebooks 01 and 02.

In [ ]:
from src.dataset import load_splits

train_df, val_df, test_df = load_splits("data")
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

## 2. Classical baselines

Fast to train (seconds), so we just retrain them here instead of loading from notebook 01.

In [ ]:
from src.models.classical_ml import ClassicalDecisionModel

rf = ClassicalDecisionModel("rf").fit(train_df)
xgb = ClassicalDecisionModel("xgb").fit(train_df)

## 3. BERT baseline

One `bert-base-uncased` encoder with 12 output logits, sliced into three heads (6 routes + 4 priorities + 2 yes/no),
trained with the sum of three cross-entropy losses. The metadata is written into the text as a prefix, e.g.
`tier: pro | channel: chat | urgency: 0.82 | failures: 2 [SEP] my app keeps crashing`.
About 1–2 minutes per epoch on a T4.

In [ ]:
from src.models.bert_baseline import BertDecisionModel

bert = BertDecisionModel("bert-base-uncased", epochs=3, batch_size=32, lr=3e-5, device=DEVICE).fit(train_df)

## 4. Laya checkpoints

The fine-tuned checkpoint comes from notebook 02. If this is a fresh runtime, point `FINETUNED_DIR` at the copy you
saved to Google Drive (set `LOAD_FROM_DRIVE = True`). If no fine-tuned checkpoint is found, that arm is skipped.

In [ ]:
from src.models.laya_model import LayaDecisionModel, BASE_MODEL_ID, TYPED_DECISIONS_MODEL_ID

LOAD_FROM_DRIVE = False
FINETUNED_DIR = "artifacts/laya-finetuned"

if LOAD_FROM_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    FINETUNED_DIR = "/content/drive/MyDrive/typed-decision-models/laya-finetuned"

laya_models = [
    LayaDecisionModel(BASE_MODEL_ID, device=DEVICE).load(),
    LayaDecisionModel(TYPED_DECISIONS_MODEL_ID, device=DEVICE).load(),
]
if os.path.exists(os.path.join(FINETUNED_DIR, "rl_agent_config.json")):
    laya_models.append(LayaDecisionModel(FINETUNED_DIR, name="laya:finetuned", device=DEVICE).load())
else:
    print("no fine-tuned Laya found at", FINETUNED_DIR, "-> skipping that arm (run notebook 02 first)")
[m.name for m in laya_models]

## 5. Run the benchmark

One call, all models. Latency is measured on single tickets (batch size 1), which is how a router is called at runtime.

In [ ]:
from src.interface import run_benchmark

models = [rf, xgb, bert] + laya_models
results = run_benchmark(models, test_df, n_latency_runs=100)
results.round(3)

Save as CSV, and as a Markdown table that can be pasted straight into the README results section.

In [ ]:
os.makedirs("artifacts", exist_ok=True)
results.to_csv("artifacts/results_full.csv")

cols = ["exact_match", "route_acc", "route_f1", "priority_f1", "requires_f1", "size_mb", "p50_ms", "p95_ms"]
table = results[cols].round(3)
try:
    text = table.to_markdown()      # needs the small `tabulate` package
except ImportError:
    text = table.to_string()
with open("artifacts/results_full.md", "w") as f:
    f.write(text)
print(open("artifacts/results_full.md").read())

## 6. Charts

Each model keeps one colour across every chart. The table above is the exact-numbers view; the charts are for
spotting the shape of the trade-off.

In [ ]:
# One fixed colour per model (validated categorical palette, colourblind-safe for adjacent pairs).
# Colour follows the model, never its rank, so every chart reads the same way.
MODEL_COLORS = {
    "random_forest": "#2a78d6",
    "xgboost": "#eb6834",
    "bert:bert-base-uncased": "#1baf7a",
    "laya:laya": "#eda100",
    "laya:laya-typed-decisions": "#e87ba4",
    "laya:finetuned": "#008300",
}
TEXT_COLOR = "#3d3d3a"
GRID_COLOR = "#e4e3dd"

import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRID_COLOR, "axes.labelcolor": TEXT_COLOR,
                     "xtick.color": TEXT_COLOR, "ytick.color": TEXT_COLOR, "axes.titlecolor": TEXT_COLOR})

model_order = list(results.index)
bar_colors = [MODEL_COLORS.get(m, "#898781") for m in model_order]


def tidy(ax):
    """Recessive axes: no top/right frame, light vertical grid behind the bars."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)

### Quality

Four small charts that share the model axis, one per metric, rather than a single crowded grouped-bar chart. Each bar
is labelled with its value.

In [ ]:
quality = [("exact_match", "Exact match (all 3 right)"), ("route_f1", "Route macro-F1"),
           ("priority_f1", "Priority macro-F1"), ("requires_f1", "Requires-human macro-F1")]

fig, axes = plt.subplots(1, len(quality), figsize=(17, 0.55 * len(model_order) + 1.4), sharey=True)
for ax, (metric, title) in zip(axes, quality):
    values = results[metric].values
    ax.barh(model_order, values, color=bar_colors, height=0.62)
    for y, v in enumerate(values):
        ax.text(v + 0.015, y, f"{v:.2f}", va="center", fontsize=9, color=TEXT_COLOR)
    ax.set_xlim(0, 1.12)
    ax.set_title(title, loc="left", fontsize=11)
    tidy(ax)
axes[0].invert_yaxis()   # first model on top
fig.tight_layout()
plt.show()

### Latency (p50 → p95)

Dot = p50 (typical request), end of the line = p95 (slow tail). **Log scale**, because trees and transformers
differ by orders of magnitude. Remember: trees ran on CPU, transformers on GPU.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.55 * len(model_order) + 1.2))
for y, model in enumerate(model_order):
    p50, p95 = results.loc[model, "p50_ms"], results.loc[model, "p95_ms"]
    color = MODEL_COLORS.get(model, "#898781")
    ax.plot([p50, p95], [y, y], color=color, linewidth=2, solid_capstyle="round")
    ax.scatter([p50], [y], s=70, color=color, edgecolor="white", linewidth=2, zorder=3)
    ax.text(p95 * 1.12, y, f"p50 {p50:.1f} · p95 {p95:.1f} ms", va="center", fontsize=9, color=TEXT_COLOR)
ax.set_yticks(range(len(model_order)), model_order)
ax.set_xscale("log")
ax.set_xlim(results["p50_ms"].min() / 2, results["p95_ms"].max() * 12)
ax.set_xlabel("milliseconds per ticket (log scale)")
ax.set_title("Single-ticket latency", loc="left", fontsize=11)
ax.invert_yaxis()
tidy(ax)
fig.tight_layout()
plt.show()

### Model size

Also log scale. Tree size depends on the data (deeper trees → bigger pickles); transformer size is fixed by the
parameter count (4 bytes per parameter in fp32).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.55 * len(model_order) + 1.2))
sizes = results["size_mb"].values
ax.barh(model_order, sizes, color=bar_colors, height=0.62)
for y, v in enumerate(sizes):
    ax.text(v * 1.12, y, f"{v:,.0f} MB", va="center", fontsize=9, color=TEXT_COLOR)
ax.set_xscale("log")
ax.set_xlim(sizes[sizes > 0].min() / 2, sizes.max() * 8)
ax.set_title("Model size (MB, log scale)", loc="left", fontsize=11)
ax.invert_yaxis()
tidy(ax)
fig.tight_layout()
plt.show()

### Quality vs latency

The real decision for a router: how much accuracy do you buy per millisecond? Upper-left is best.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for model in model_order:
    x, y = results.loc[model, "p50_ms"], results.loc[model, "exact_match"]
    ax.scatter([x], [y], s=90, color=MODEL_COLORS.get(model, "#898781"), edgecolor="white", linewidth=2, zorder=3)
    ax.annotate(model, (x, y), xytext=(8, 4), textcoords="offset points", fontsize=9, color=TEXT_COLOR)
ax.set_xscale("log")
ax.set_xlabel("p50 latency, ms (log scale)")
ax.set_ylabel("exact match")
ax.set_ylim(0, 1.05)
ax.set_title("Exact match vs p50 latency", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color=GRID_COLOR, linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 7. Reading the results (fill in after running)

- **Best exact match:** _TODO_
- **Fastest model within ~2 points of the best:** _TODO_
- **Zero-shot Laya vs fine-tuned Laya:** _TODO_
- **Does BERT's text-prefix trick for metadata keep up with the trees on priority?** _TODO_

Copy `artifacts/results_full.md` into the README results table.

Next: `04_langgraph_decision_router.ipynb` plugs a trained model into an agent workflow.